# EconoCausal — Improved Causal Retail Dataset

## Project: Dynamic Pricing via Double Machine Learning

This notebook generates an improved synthetic retail dataset for the EconoCausal project.

The purpose of this dataset is to support causal inference and dynamic pricing using Double Machine Learning (DML).

Unlike a simple predictive dataset, the data is designed to contain:

- Pre-treatment customer characteristics
- Confounded discount assignment
- Heterogeneous customer responses to discounts
- Multiple treatment levels: $0, $10, and $20
- Customer-level counterfactual purchase probabilities
- An economic profit model for policy optimization
- Hidden ground-truth treatment effects for later model validation

The final dataset will be used in the causal modeling stage to estimate treatment effects and determine which discount is economically optimal for different customers.

## Why an Improved Dataset Is Needed

The original synthetic dataset was useful for demonstrating the basic DML workflow, but it had some limitations for a realistic dynamic pricing problem.

### Limitations of the original dataset

- Discount assignment was randomized, so customer characteristics did not influence treatment assignment.
- The treatment effect increased proportionally with the discount level.
- The $20 discount was therefore consistently more effective than the $10 discount.
- The dataset did not contain a sufficiently realistic economic trade-off between discount cost and increased purchase probability.

These limitations made the pricing policy relatively simple because the highest discount was generally the best treatment.

### Improvements in this dataset

The improved dataset addresses these limitations by introducing:

1. **Confounded treatment assignment** — customer characteristics influence whether a customer receives $0, $10, or $20.
2. **Heterogeneous treatment effects** — different customers respond differently to discounts.
3. **Diminishing returns** — the additional effect of increasing the discount from $10 to $20 is smaller than the initial effect of $0 to $10.
4. **Economic optimization** — discount decisions are evaluated using expected profit rather than purchase probability alone.
5. **Ground-truth counterfactuals** — the true treatment effects and optimal discount are retained separately for validating the causal models.

This creates a more challenging and realistic environment for evaluating Double Machine Learning and customer-level treatment policies.

## Dataset Design

The improved dataset contains 100,000 synthetic customers.

The dataset is designed around three discount treatment levels:

- **$0 discount** — no treatment
- **$10 discount** — moderate treatment
- **$20 discount** — high treatment

Each customer has a set of pre-treatment characteristics that may influence both their purchase behavior and the discount they receive.

### Dataset components

The dataset contains:

- **12 pre-treatment customer features**
- **1 treatment variable:** `discount`
- **1 observed outcome:** `purchase`
- **1 customer identifier:** `customer_id`

The true treatment effects and the economically optimal discount are generated separately and stored as ground truth. They are not provided to the causal model during training.

This separation allows the estimated treatment effects from Double Machine Learning to be compared against known ground-truth effects during model validation.

## Reproducibility and Dataset Size

A fixed random seed is used so that the synthetic dataset can be regenerated with the same results.

The dataset contains 100,000 customers, providing a sufficiently large sample for causal modeling and subgroup-level analysis.

Using a fixed seed also makes the experiments reproducible when comparing different causal estimation methods.

In [1]:
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

N = 100_000

print(f"Random seed: {RANDOM_SEED}")
print(f"Number of customers: {N:,}")

Random seed: 42
Number of customers: 100,000


## Pre-Treatment Customer Characteristics

The dataset contains 12 customer characteristics measured before the discount is assigned.

These variables represent customer behavior, engagement, purchasing history, and customer relationship characteristics.

| Feature | Description |
|---|---|
| `age` | Customer age |
| `past_purchases` | Number of previous purchases |
| `product_views` | Number of product views |
| `website_visits` | Number of website visits |
| `avg_order_value` | Historical average order value |
| `days_since_last_purchase` | Days since the customer's previous purchase |
| `loyalty_score` | Customer loyalty score |
| `cart_additions` | Number of products added to cart |
| `email_engagement` | Customer engagement with emails |
| `customer_tenure` | Length of the customer relationship |
| `product_interest_score` | Interest in the products |
| `previous_discount_usage` | Historical use of discounts |

These variables are treated as **pre-treatment variables** because their values are determined before the current discount is assigned.

They can therefore be used as covariates when estimating the causal effect of the discount.

In [2]:
# Pre-treatment customer characteristics

df = pd.DataFrame({
    "customer_id": [f"C{i:06d}" for i in range(1, N + 1)],

    "age": rng.integers(18, 71, size=N),

    "past_purchases": rng.poisson(
        lam=5, size=N
    ).clip(0, 30),

    "product_views": rng.poisson(
        lam=12, size=N
    ).clip(0, 100),

    "website_visits": rng.poisson(
        lam=8, size=N
    ).clip(0, 80),

    "avg_order_value": np.clip(
        rng.normal(1200, 500, size=N),
        100,
        5000
    ).round(2),

    "days_since_last_purchase": rng.integers(
        1, 181, size=N
    ),

    "loyalty_score": rng.integers(
        1, 101, size=N
    ),

    "cart_additions": rng.poisson(
        lam=3, size=N
    ).clip(0, 30),

    "email_engagement": rng.integers(
        0, 101, size=N
    ),

    "customer_tenure": rng.integers(
        1, 61, size=N
    ),

    "product_interest_score": rng.integers(
        1, 101, size=N
    ),

    "previous_discount_usage": rng.integers(
        0, 11, size=N
    )
})

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (100000, 13)


,customer_id,age,past_purchases,product_views,website_visits,avg_order_value,days_since_last_purchase,loyalty_score,cart_additions,email_engagement,customer_tenure,product_interest_score,previous_discount_usage
0,C000001,22,6,15,12,1057.19,132,81,1,99,11,44,1
1,C000002,59,5,6,7,2105.66,165,38,1,50,22,24,8
2,C000003,52,8,12,6,1422.69,170,49,6,10,11,28,5
3,C000004,41,3,15,14,857.64,54,14,5,97,11,27,10
4,C000005,40,6,14,6,323.18,33,31,3,74,21,19,9


## Confounded Treatment Assignment

In a real pricing system, discounts are unlikely to be assigned completely at random.

Businesses may target customers differently based on their previous behavior, engagement, purchase recency, and likelihood of responding to an offer.

To represent this in the synthetic data, a latent treatment score is constructed from pre-treatment customer characteristics.

Customers with different characteristics therefore have different probabilities of receiving:

- $0 discount
- $10 discount
- $20 discount

This creates **treatment confounding**, meaning that the characteristics of customers receiving different discount levels are systematically different.

This is important for causal inference because the model must account for these pre-treatment differences when estimating the effect of the discount on purchase.

In [4]:
# Confounded treatment assignment
# Customer characteristics influence which discount they receive.

discount_score = (
    0.035 * df["days_since_last_purchase"]
    + 0.025 * df["product_interest_score"]
    + 0.020 * df["email_engagement"]
    + 0.030 * df["previous_discount_usage"]
    + 0.020 * df["cart_additions"]
    - 0.020 * df["loyalty_score"]
    + rng.normal(0, 2.0, N)
)

# Standardize the latent treatment score
score_z = (
    discount_score - discount_score.mean()
) / discount_score.std()

# Multinomial treatment logits
logit_0 = np.zeros(N)

logit_10 = (
    0.7 * score_z + 0.2
)

logit_20 = (
    0.9 * score_z - 0.2
)

# Softmax probabilities
logits = np.column_stack([
    logit_0,
    logit_10,
    logit_20
])

exp_logits = np.exp(
    logits - logits.max(axis=1, keepdims=True)
)

probabilities = (
    exp_logits / exp_logits.sum(axis=1, keepdims=True)
)

# Assign $0, $10 or $20
df["discount"] = [
    rng.choice([0, 10, 20], p=probabilities[i])
    for i in range(N)
]

print("Discount distribution:")
print(df["discount"].value_counts().sort_index())

print("\nDiscount proportions:")
print(df["discount"].value_counts(normalize=True).sort_index())

Discount distribution:
discount
0     34831
10    38095
20    27074
Name: count, dtype: int64

Discount proportions:
discount
0     0.34831
10    0.38095
20    0.27074
Name: proportion, dtype: float64


## Heterogeneous Treatment Effects

Customers do not necessarily respond to discounts in the same way.

For example, a highly interested customer who frequently adds products to their cart may respond more strongly to a discount than a customer with low product interest.

To represent this heterogeneity, the synthetic data generates a customer-specific treatment effect based on pre-treatment characteristics.

The treatment effects are defined as:

- `true_effect_10` — the true increase in purchase probability caused by a $10 discount compared with no discount.
- `additional_effect_20` — the additional increase caused by increasing the discount from $10 to $20.
- `true_effect_20` — the total effect of a $20 discount compared with no discount.

The second $10 is intentionally given a smaller incremental effect to represent **diminishing returns** from increasing the discount.

These true effects are generated only for ground-truth validation and are not provided as inputs to the causal model.

In [5]:
# Baseline purchase probability and heterogeneous treatment effects

# 1. Baseline purchase tendency without any discount
baseline_logit = (
    -3.0
    + 0.025 * df["loyalty_score"]
    + 0.030 * df["product_interest_score"]
    + 0.080 * df["past_purchases"]
    + 0.020 * df["website_visits"]
    + 0.025 * df["cart_additions"]
    - 0.010 * df["days_since_last_purchase"]
    + 0.010 * df["email_engagement"]
)

baseline_probability = 1 / (
    1 + np.exp(-baseline_logit)
)

# 2. Customer-specific responsiveness to discounts
base_response = (
    0.0015 * df["product_interest_score"]
    + 0.0010 * df["loyalty_score"]
    + 0.0020 * df["cart_additions"]
    + 0.0008 * df["email_engagement"]
)

# 3. True effect of $10 discount
true_effect_10 = base_response

# 4. True effect of increasing from $10 to $20
#    Diminishing returns: the second $10 is not equally effective.
additional_effect_20 = (
    0.45 * base_response
    - 0.000015 * df["avg_order_value"]
    + 0.0005 * df["product_interest_score"]
)

# Prevent negative incremental effects
additional_effect_20 = additional_effect_20.clip(lower=0.005)

# 5. True effect of $20 relative to $0
true_effect_20 = (
    true_effect_10
    + additional_effect_20
)

# 6. Counterfactual purchase probabilities
prob_0 = baseline_probability
prob_10 = (
    baseline_probability + true_effect_10
).clip(0.01, 0.99)

prob_20 = (
    baseline_probability + true_effect_20
).clip(0.01, 0.99)

print("Average baseline purchase probability:",
      round(prob_0.mean(), 4))

print("Average true effect of $10:",
      round(true_effect_10.mean(), 4))

print("Average true effect of $20:",
      round(true_effect_20.mean(), 4))

print("Average incremental effect of $20 vs $10:",
      round(additional_effect_20.mean(), 4))

Average baseline purchase probability: 0.5027
Average true effect of $10: 0.1727
Average true effect of $20: 0.2577
Average incremental effect of $20 vs $10: 0.0851


## Observed Purchase Outcome

The observed purchase outcome is generated from the purchase probability corresponding to the discount actually assigned to each customer.

For each customer:

- Customers receiving $0 use `prob_0`.
- Customers receiving $10 use `prob_10`.
- Customers receiving $20 use `prob_20`.

A Bernoulli random draw is then used to generate the binary `purchase` outcome:

- `0` — customer does not purchase
- `1` — customer makes a purchase

This represents the observed outcome that would be available in a real-world dataset.

The counterfactual outcomes — what the same customer would have done under the other discount levels — are not directly observed. These counterfactuals are retained separately as ground truth for later validation of the causal models.

In [6]:
# Generate observed purchase outcome based on the assigned discount

observed_probability = np.select(
    [
        df["discount"] == 0,
        df["discount"] == 10,
        df["discount"] == 20
    ],
    [
        prob_0,
        prob_10,
        prob_20
    ]
)

df["purchase"] = rng.binomial(
    1,
    observed_probability
)

print("Purchase rate:", round(df["purchase"].mean(), 4))

print("\nPurchase rate by discount:")
print(
    df.groupby("discount")["purchase"]
      .mean()
      .round(4)
)

Purchase rate: 0.6188

Purchase rate by discount:
discount
0     0.5189
10    0.6500
20    0.7036
Name: purchase, dtype: float64


## Economic Profit Model

Purchase probability alone is not sufficient for determining the best discount.

A higher discount may increase the probability of purchase, but it also reduces the amount of revenue retained from each purchase.

To capture this trade-off, an expected profit model is introduced.

A gross margin rate of **30%** is assumed for the synthetic business.

For each discount level, expected profit is calculated as:

**Expected Profit = Purchase Probability × (Expected Gross Profit Before Discount − Discount)**

The model calculates expected profit under all three counterfactual discount levels:

- `profit_0` — expected profit with no discount
- `profit_10` — expected profit with a $10 discount
- `profit_20` — expected profit with a $20 discount

This allows the optimal pricing decision to be based on **economic value rather than purchase probability alone**.

In [7]:
# Economic assumptions and customer-level profit under each discount

# Assume the business keeps 30% of the order value as gross margin
MARGIN_RATE = 0.30

# Expected profit per purchase before discount
profit_before_discount = (
    df["avg_order_value"] * MARGIN_RATE
)

# Expected profit under each counterfactual discount
profit_0 = (
    prob_0 * profit_before_discount
)

profit_10 = (
    prob_10 * (profit_before_discount - 10)
)

profit_20 = (
    prob_20 * (profit_before_discount - 20)
)

print("Average expected profit:")
print("$0  :", round(profit_0.mean(), 2))
print("$10 :", round(profit_10.mean(), 2))
print("$20 :", round(profit_20.mean(), 2))

Average expected profit:
$0  : 181.3
$10 : 229.38
$20 : 240.46


## True Optimal Discount

The economically optimal discount is determined separately for each customer.

For every customer, the expected profit under $0, $10, and $20 discounts is compared.

The discount that produces the highest expected profit is stored as:

`true_optimal_discount`

This creates the ground-truth pricing policy for the synthetic dataset.

The resulting policy can contain three customer groups:

- Customers for whom **$0** is optimal
- Customers for whom **$10** is optimal
- Customers for whom **$20** is optimal

This ground truth will later be used to evaluate whether the causal model can correctly identify the economically preferred treatment for individual customers.

In [13]:
# True optimal discount for each customer

profit_matrix = np.column_stack([
    profit_0,
    profit_10,
    profit_20
])

discount_options = np.array([0, 10, 20])

best_policy_index = np.argmax(
    profit_matrix,
    axis=1
)

df["true_optimal_discount"] = (
    discount_options[best_policy_index]
)

print("True optimal discount distribution:")
print(
    df["true_optimal_discount"]
      .value_counts()
      .sort_index()
)

print("\nTrue optimal discount proportions:")
print(
    df["true_optimal_discount"]
      .value_counts(normalize=True)
      .sort_index()
      .round(4)
)

True optimal discount distribution:
true_optimal_discount
0      2045
10    22494
20    75461
Name: count, dtype: int64

True optimal discount proportions:
true_optimal_discount
0     0.0204
10    0.2249
20    0.7546
Name: proportion, dtype: float64


In [14]:
print("Average expected profit by optimal discount:")
print(
    pd.DataFrame({
        "profit_0": profit_0,
        "profit_10": profit_10,
        "profit_20": profit_20,
        "optimal_discount": df["true_optimal_discount"]
    })
    .groupby("optimal_discount")[["profit_0", "profit_10", "profit_20"]]
    .mean()
    .round(2)
)

Average expected profit by optimal discount:
                  profit_0  profit_10  profit_20
optimal_discount                                
0                    79.32      76.60      68.60
10                  283.02     323.54     314.92
20                  153.74     205.45     222.93


## Ground Truth and Modeling Data Separation

The synthetic data generator knows the true treatment effects because they are used to construct the counterfactual outcomes.

However, a causal model should not have access to these values during estimation.

Therefore, the generated data is separated into two datasets:

### Modeling dataset

`causal_retail_data.csv`

Contains only variables that would be available to the causal model:

- Customer characteristics
- Assigned discount
- Observed purchase outcome

### Ground-truth dataset

`causal_retail_ground_truth.csv`

Contains the known counterfactual information used only for validation:

- `true_effect_10`
- `true_effect_20`
- `additional_effect_20`
- `true_optimal_discount`

This separation prevents information leakage and allows the estimated treatment effects and pricing policy to be evaluated against known ground truth after model training.

In [9]:
# Preserve ground truth separately for later model validation

ground_truth = df[
    [
        "customer_id",
        "true_optimal_discount"
    ]
].copy()

ground_truth["true_effect_10"] = true_effect_10
ground_truth["true_effect_20"] = true_effect_20
ground_truth["additional_effect_20"] = additional_effect_20

# Keep only variables that would actually be available to the causal model
modeling_df = df.drop(
    columns=["true_optimal_discount"]
).copy()

print("Modeling dataset shape:", modeling_df.shape)
print("Ground-truth dataset shape:", ground_truth.shape)

print("\nModeling columns:")
print(modeling_df.columns.tolist())

print("\nGround-truth columns:")
print(ground_truth.columns.tolist())

Modeling dataset shape: (100000, 15)
Ground-truth dataset shape: (100000, 5)

Modeling columns:
['customer_id', 'age', 'past_purchases', 'product_views', 'website_visits', 'avg_order_value', 'days_since_last_purchase', 'loyalty_score', 'cart_additions', 'email_engagement', 'customer_tenure', 'product_interest_score', 'previous_discount_usage', 'discount', 'purchase']

Ground-truth columns:
['customer_id', 'true_optimal_discount', 'true_effect_10', 'true_effect_20', 'additional_effect_20']


## Saving the Datasets

The final datasets are saved as separate CSV files in the project's `data/` directory.

### `causal_retail_data.csv`

This is the main dataset used for causal modeling. It contains the pre-treatment features, assigned discount, and observed purchase outcome.

### `causal_retail_ground_truth.csv`

This contains the hidden treatment effects and the true optimal discount. It is reserved for model evaluation and validation.

Keeping these files separate ensures that the causal models cannot accidentally use ground-truth information during training.

In [10]:
# Save the improved datasets

modeling_df.to_csv(
    "../data/causal_retail_data.csv",
    index=False
)

ground_truth.to_csv(
    "../data/causal_retail_ground_truth.csv",
    index=False
)

print("Saved:")
print("../data/causal_retail_data.csv")
print("../data/causal_retail_ground_truth.csv")

Saved:
../data/causal_retail_data.csv
../data/causal_retail_ground_truth.csv


## Final Data-Quality Validation

Before using the dataset for causal modeling, basic data-quality checks are performed.

The validation checks include:

- Missing values
- Duplicate customer IDs
- Valid discount values
- Valid binary purchase outcomes
- Numerical feature ranges and summary statistics

These checks help ensure that the generated dataset is structurally valid and ready for the causal modeling stage.

In [15]:
# Final data-quality checks

print("Missing values:")
print(modeling_df.isnull().sum().sum())

print("\nDuplicate customer IDs:")
print(modeling_df["customer_id"].duplicated().sum())

print("\nDiscount values:")
print(sorted(modeling_df["discount"].unique()))

print("\nPurchase values:")
print(sorted(modeling_df["purchase"].unique()))

print("\nNumeric summary:")
display(modeling_df.describe())

Missing values:
0

Duplicate customer IDs:
0

Discount values:
[np.int64(0), np.int64(10), np.int64(20)]

Purchase values:
[np.int64(0), np.int64(1)]

Numeric summary:


,age,past_purchases,product_views,website_visits,avg_order_value,days_since_last_purchase,loyalty_score,cart_additions,email_engagement,customer_tenure,product_interest_score,previous_discount_usage,discount,purchase
count,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,43.99190,4.997130,12.005450,7.997930,1203.034403,90.580310,50.562620,3.000110,50.013840,30.529520,50.720150,4.985620,9.224300,0.618850
std,15.29454,2.226852,3.466304,2.824303,494.296136,52.061714,28.828127,1.730445,29.196099,17.291502,28.931338,3.162289,7.829681,0.485672
min,18.00000,0.000000,0.000000,0.000000,100.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,31.00000,3.000000,10.000000,6.000000,861.257500,45.000000,26.000000,2.000000,25.000000,16.000000,26.000000,2.000000,0.000000,0.000000
50%,44.00000,5.000000,12.000000,8.000000,1198.700000,90.000000,50.000000,3.000000,50.000000,31.000000,51.000000,5.000000,10.000000,1.000000
75%,57.00000,6.000000,14.000000,10.000000,1539.825000,136.000000,76.000000,4.000000,75.000000,46.000000,76.000000,8.000000,20.000000,1.000000
max,70.00000,19.000000,30.000000,22.000000,3552.080000,180.000000,100.000000,13.000000,100.000000,60.000000,100.000000,10.000000,20.000000,1.000000


## Treatment Group Balance and Confounding Check

The distribution of customers across the three discount levels is examined to understand the treatment assignment.

The average values of selected pre-treatment characteristics are also compared across discount groups.

If these characteristics differ systematically between groups, it provides evidence that discount assignment is not random and that treatment confounding is present.

These differences are important for the later causal modeling stage because the model must account for pre-treatment customer characteristics when estimating treatment effects.

In [12]:
# Check treatment group sizes and key feature differences

print("Discount counts:")
print(modeling_df["discount"].value_counts().sort_index())

print("\nDiscount proportions:")
print(
    modeling_df["discount"]
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nAverage customer characteristics by discount:")
display(
    modeling_df.groupby("discount")[
        [
            "age",
            "loyalty_score",
            "product_interest_score",
            "previous_discount_usage",
            "days_since_last_purchase"
        ]
    ].mean().round(2)
)

Discount counts:
discount
0     34831
10    38095
20    27074
Name: count, dtype: int64

Discount proportions:
discount
0     0.3483
10    0.3810
20    0.2707
Name: proportion, dtype: float64

Average customer characteristics by discount:


,age,loyalty_score,product_interest_score,previous_discount_usage,days_since_last_purchase
discount,,,,,
0,43.89,53.05,47.45,4.94,75.91
10,44.09,49.79,51.95,5.01,96.38
20,43.98,48.45,53.20,5.00,101.30
